<a href="https://colab.research.google.com/github/ced-sys/AI-N-ML/blob/main/Adaptation_Atlas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import ee
import folium
from folium import plugins
from google.colab import auth

auth.authenticate_user()
ee.Initialize(project='ee-victorcedric665')

In [38]:
dataset=ee.ImageCollection('MODIS/061/MCD15A3H') \
.select('Fpar') \
.filterDate('2023-01-01', '2023-12-31')

In [39]:
fpar_image=dataset.first()

In [40]:
gaul=ee.FeatureCollection('FAO/GAUL/2015/level1')
kenya=gaul.filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))

In [41]:
import json

fpar_kenya = fpar_image.clip(kenya)

fpar_vis = {
    'min': 0.0,
    'max': 100.0,
    'palette': ['e1e4b4', '999d60', '2ec409', '0a4b06']
}

Map = folium.Map(location=[0.0236, 37.9062], zoom_start=6, tiles='cartodbpositron')

# Add FPAR layer
map_id_dict = ee.Image(fpar_kenya).getMapId(fpar_vis)
folium.TileLayer(
    tiles=map_id_dict['tile_fetcher'].url_format,
    attr='Google Earth Engine | MODIS FPAR',
    overlay=True,
    name='MODIS FPAR (2023)',
).add_to(Map)

# Add Kenya boundary - fixed method
kenya_geojson = kenya.getInfo()
folium.GeoJson(
    kenya_geojson,
    name='Kenya Boundary',
    style_function=lambda x: {'color': 'black', 'fill': False, 'weight': 1}
).add_to(Map)

folium.LayerControl().add_to(Map)

Map

In [42]:
import ee
import folium
from branca.colormap import LinearColormap

# Load CHIRPS daily precipitation dataset
dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filter(ee.Filter.date('2018-05-01', '2018-05-03'))

precipitation = dataset.select('precipitation')

# Load Kenya boundaries from GAUL
gaul = ee.FeatureCollection('FAO/GAUL/2015/level1')
kenya = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))

# Get mean precipitation and clip to Kenya
precip_mean = precipitation.mean()
precip_kenya = precip_mean.clip(kenya)

# Visualization parameters
precipitation_vis = {
    'min': 1,
    'max': 17,
    'palette': ['001137', '0aab1e', 'e7eb05', 'ff4a2d', 'e90000']
}

# Create map centered on Kenya
Map = folium.Map(location=[0.0236, 37.9062], zoom_start=6, tiles='cartodbpositron')

# Add precipitation layer
map_id_dict = ee.Image(precip_kenya).getMapId(precipitation_vis)
folium.TileLayer(
    tiles=map_id_dict['tile_fetcher'].url_format,
    attr='Google Earth Engine | CHIRPS Daily Precipitation',
    overlay=True,
    name='Precipitation (2018-05-01 to 2018-05-03)',
).add_to(Map)

# Add Kenya boundary
kenya_geojson = kenya.getInfo()
folium.GeoJson(
    kenya_geojson,
    name='Kenya Boundary',
    style_function=lambda x: {'color': 'black', 'fill': False, 'weight': 1}
).add_to(Map)

# Add layer control
folium.LayerControl().add_to(Map)

# Add color bar
colormap = LinearColormap(
    colors=['#001137', '#0aab1e', '#e7eb05', '#ff4a2d', '#e90000'],
    vmin=1,
    vmax=17,
    caption='Precipitation (mm/day)'
)
colormap.add_to(Map)

Map

In [43]:
import ee
import folium
from branca.colormap import StepColormap

# Load GFSAD1000 cropland extent dataset
dataset = ee.Image('USGS/GFSAD1000_V1')
crop_mask = dataset.select('landcover')

# Load Kenya boundaries from GAUL
gaul = ee.FeatureCollection('FAO/GAUL/2015/level1')
kenya = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))

# Clip to Kenya
crop_mask_kenya = crop_mask.clip(kenya)

# Visualization parameters
crop_mask_vis = {
    'min': 0.0,
    'max': 5.0,
    'palette': ['black', 'orange', 'brown', '02a50f', 'green', 'yellow']
}

# Create map centered on Kenya
Map = folium.Map(location=[0.0236, 37.9062], zoom_start=6, tiles='cartodbpositron')

# Add crop mask layer
map_id_dict = ee.Image(crop_mask_kenya).getMapId(crop_mask_vis)
folium.TileLayer(
    tiles=map_id_dict['tile_fetcher'].url_format,
    attr='Google Earth Engine | GFSAD1000 Cropland Extent',
    overlay=True,
    name='Crop Mask',
).add_to(Map)

# Add Kenya boundary
kenya_geojson = kenya.getInfo()
folium.GeoJson(
    kenya_geojson,
    name='Kenya Boundary',
    style_function=lambda x: {'color': 'black', 'fill': False, 'weight': 1}
).add_to(Map)

# Add layer control
folium.LayerControl().add_to(Map)

# Add legend for land cover classes
legend_labels = {
    0: 'Non-croplands',
    1: 'Croplands: irrigation major',
    2: 'Croplands: irrigation minor',
    3: 'Croplands: rainfed',
    4: 'Croplands: rainfed, minor fragments',
    5: 'Croplands: rainfed, very minor fragments'
}

colormap = StepColormap(
    colors=['#000000', '#ffa500', '#a52a2a', '#02a50f', '#008000', '#ffff00'],
    vmin=0,
    vmax=5,
    index=[0, 1, 2, 3, 4, 5, 6],
    caption='GFSAD1000 Land Cover Classification'
)
colormap.add_to(Map)

Map

In [44]:
# ============================================================================
# RAINFALL RELIABILITY EXPLORER - COMPLETE WORKFLOW
# Adaptation Atlas Hackathon
# ============================================================================
# This notebook processes CHIRPS, MODIS, and Cropland data to compute
# rainfall reliability metrics for Kenya at the county level.
# ============================================================================

# %% [markdown]
# ## 1️⃣ Setup and Authentication

# %%
import ee
import folium
from folium import plugins
from branca.colormap import LinearColormap, StepColormap
import pandas as pd
from google.colab import auth
import json

# Authenticate and initialize Earth Engine
auth.authenticate_user()
ee.Initialize(project='ee-victorcedric665')

print("✅ Earth Engine initialized successfully!")

# %% [markdown]
# ## 2️⃣ Define AOI and Administrative Units

# %%
# Load Kenya administrative boundaries (county level)
gaul = ee.FeatureCollection('FAO/GAUL/2015/level1')
kenya_admin1 = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))

# Get Kenya country boundary for clipping
kenya_boundary = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))

# Get county count
county_count = kenya_admin1.size().getInfo()
print(f"📍 Loaded {county_count} counties in Kenya")

# Visualize counties
Map = folium.Map(location=[0.0236, 37.9062], zoom_start=6, tiles='cartodbpositron')
kenya_geojson = kenya_admin1.getInfo()
folium.GeoJson(
    kenya_geojson,
    name='Kenya Counties',
    style_function=lambda x: {'color': 'blue', 'fillColor': 'lightblue',
                              'fillOpacity': 0.2, 'weight': 1}
).add_to(Map)
folium.LayerControl().add_to(Map)
Map

# %% [markdown]
# ## 3️⃣ Load and Prepare Datasets

# %%
# Define time range for analysis
start_date = '2010-01-01'
end_date = '2023-12-31'

print(f"📅 Analysis period: {start_date} to {end_date}")

# Load CHIRPS daily rainfall data
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterBounds(kenya_boundary) \
    .filterDate(start_date, end_date) \
    .select('precipitation')

print(f"🌧️  CHIRPS: {chirps.size().getInfo()} daily images loaded")

# Load MODIS NDVI data (alternatively use FPAR)
modis_ndvi = ee.ImageCollection('MODIS/061/MOD13A2') \
    .filterBounds(kenya_boundary) \
    .filterDate(start_date, end_date) \
    .select('NDVI')

print(f"🌿 MODIS NDVI: {modis_ndvi.size().getInfo()} images loaded")

# Load cropland mask
cropland_dataset = ee.Image('USGS/GFSAD1000_V1').select('landcover')
# Create binary cropland mask (values 1-5 are cropland)
cropland_mask = cropland_dataset.gte(1).And(cropland_dataset.lte(5))

print("🌾 Cropland mask loaded")

# %% [markdown]
# ## 4️⃣ Visualize Raw Data (Quality Check)

# %%
# Visualize mean annual rainfall
mean_rainfall = chirps.filterDate('2022-01-01', '2022-12-31').sum().clip(kenya_boundary)

rainfall_vis = {
    'min': 0,
    'max': 1500,
    'palette': ['#ffffcc', '#c7e9b4', '#7fcdbb', '#41b6c4', '#1d91c0', '#225ea8', '#0c2c84']
}

Map2 = folium.Map(location=[0.0236, 37.9062], zoom_start=6, tiles='cartodbpositron')

map_id = mean_rainfall.getMapId(rainfall_vis)
folium.TileLayer(
    tiles=map_id['tile_fetcher'].url_format,
    attr='CHIRPS Annual Rainfall 2022',
    name='Annual Rainfall (mm)',
    overlay=True
).add_to(Map2)

# Add counties
folium.GeoJson(
    kenya_geojson,
    name='Counties',
    style_function=lambda x: {'color': 'black', 'fill': False, 'weight': 1}
).add_to(Map2)

folium.LayerControl().add_to(Map2)

# Add colorbar
colormap = LinearColormap(
    colors=['#ffffcc', '#c7e9b4', '#7fcdbb', '#41b6c4', '#1d91c0', '#225ea8', '#0c2c84'],
    vmin=0,
    vmax=1500,
    caption='Annual Rainfall (mm)'
)
colormap.add_to(Map2)

Map2

# %% [markdown]
# ## 5️⃣ Compute Seasonal Rainfall Metrics

# %%
def compute_seasonal_rainfall(year, season):
    """
    Compute seasonal rainfall for a given year and season.

    Args:
        year: Year to analyze
        season: 'MAM' (March-April-May) or 'OND' (October-November-December)

    Returns:
        ee.Image with total seasonal rainfall
    """
    if season == 'MAM':
        start = ee.Date.fromYMD(year, 3, 1)
        end = ee.Date.fromYMD(year, 5, 31)
    elif season == 'OND':
        start = ee.Date.fromYMD(year, 10, 1)
        end = ee.Date.fromYMD(year, 12, 31)
    else:
        raise ValueError("Season must be 'MAM' or 'OND'")

    seasonal_rain = chirps.filterDate(start, end).sum()
    return seasonal_rain.set('year', year).set('season', season)

# Compute seasonal rainfall for all years
years = list(range(2010, 2024))
seasons = ['MAM', 'OND']

seasonal_rainfall_list = []
for year in years:
    for season in seasons:
        img = compute_seasonal_rainfall(year, season)
        seasonal_rainfall_list.append(img)

seasonal_rainfall = ee.ImageCollection(seasonal_rainfall_list)

print(f"✅ Computed seasonal rainfall for {len(years)} years × {len(seasons)} seasons")

# %% [markdown]
# ## 6️⃣ Compute Rainfall Onset and Cessation

# %%
def compute_rainfall_onset(year, season='MAM'):
    """
    Compute rainfall onset date (simplified version).
    Onset = first day when cumulative rainfall > 20mm over 3 days
    """
    if season == 'MAM':
        start = ee.Date.fromYMD(year, 3, 1)
        end = ee.Date.fromYMD(year, 5, 31)
    else:  # OND
        start = ee.Date.fromYMD(year, 10, 1)
        end = ee.Date.fromYMD(year, 12, 31)

    # Get daily rainfall for the season
    daily_rain = chirps.filterDate(start, end)

    # Compute 3-day rolling sum
    def rolling_sum(img):
        date = img.date()
        sum_3day = chirps.filterDate(date.advance(-2, 'day'), date.advance(1, 'day')).sum()
        return sum_3day.set('system:time_start', img.get('system:time_start'))

    rolling_rainfall = daily_rain.map(rolling_sum)

    # Find first day where 3-day sum > 20mm
    # (This is a simplified approach - production code would need more sophisticated detection)
    onset_threshold = rolling_rainfall.map(lambda img: img.gte(20))

    return rolling_rainfall.mean().set('year', year).set('season', season)

print("📊 Onset computation function defined (simplified version)")

# %% [markdown]
# ## 7️⃣ Compute Length of Growing Period (LGP)

# %%
def compute_lgp_proxy(year, season='MAM'):
    """
    Compute a proxy for Length of Growing Period using NDVI.
    LGP ≈ number of days with NDVI > threshold during season
    """
    if season == 'MAM':
        start = ee.Date.fromYMD(year, 3, 1)
        end = ee.Date.fromYMD(year, 5, 31)
    else:
        start = ee.Date.fromYMD(year, 10, 1)
        end = ee.Date.fromYMD(year, 12, 31)

    # Get MODIS NDVI for season (scale 0-10000)
    ndvi_season = modis_ndvi.filterDate(start, end)

    # Count days with NDVI > 3000 (green vegetation)
    growing_days = ndvi_season.map(lambda img: img.gt(3000))
    lgp_proxy = growing_days.sum().multiply(16)  # MODIS is 16-day composite

    return lgp_proxy.set('year', year).set('season', season)

# Compute LGP proxy for all years
lgp_list = []
for year in years:
    for season in seasons:
        img = compute_lgp_proxy(year, season)
        lgp_list.append(img)

lgp_collection = ee.ImageCollection(lgp_list)

print("✅ LGP proxy computed for all years and seasons")

# %% [markdown]
# ## 8️⃣ Apply Cropland Mask

# %%
# Mask seasonal rainfall to cropland only
seasonal_rainfall_masked = seasonal_rainfall.map(lambda img: img.updateMask(cropland_mask))

# Mask LGP to cropland only
lgp_masked = lgp_collection.map(lambda img: img.updateMask(cropland_mask))

print("✅ Cropland mask applied to all datasets")

# Visualize masked data
masked_rain_2022 = seasonal_rainfall_masked \
    .filter(ee.Filter.eq('year', 2022)) \
    .filter(ee.Filter.eq('season', 'MAM')) \
    .first() \
    .clip(kenya_boundary)

Map3 = folium.Map(location=[0.0236, 37.9062], zoom_start=6, tiles='cartodbpositron')

map_id = masked_rain_2022.getMapId(rainfall_vis)
folium.TileLayer(
    tiles=map_id['tile_fetcher'].url_format,
    attr='Cropland Rainfall MAM 2022',
    name='Cropland Rainfall',
    overlay=True
).add_to(Map3)

folium.GeoJson(
    kenya_geojson,
    name='Counties',
    style_function=lambda x: {'color': 'black', 'fill': False, 'weight': 1}
).add_to(Map3)

folium.LayerControl().add_to(Map3)
Map3

# %% [markdown]
# ## 9️⃣ Compute Rainfall Variability and Trends

# %%
# Compute mean and standard deviation across years
mean_seasonal_rainfall = seasonal_rainfall_masked.mean()
std_seasonal_rainfall = seasonal_rainfall_masked.reduce(ee.Reducer.stdDev())

# Compute coefficient of variation (CV = std/mean)
cv_rainfall = std_seasonal_rainfall.divide(mean_seasonal_rainfall).multiply(100)

# Compute linear trend using linearFit
def add_time_band(img):
    """Add a time band for trend analysis"""
    date = ee.Date(img.get('system:time_start'))
    years_since_start = date.difference(ee.Date(start_date), 'year')
    return img.addBands(ee.Image(years_since_start).rename('time').float())

rainfall_with_time = seasonal_rainfall_masked.map(add_time_band)
rainfall_trend = rainfall_with_time.select(['time', 'precipitation']).reduce(ee.Reducer.linearFit())

print("✅ Computed rainfall variability and trends")

# Visualize coefficient of variation
cv_vis = {
    'min': 0,
    'max': 50,
    'palette': ['green', 'yellow', 'orange', 'red']
}

Map4 = folium.Map(location=[0.0236, 37.9062], zoom_start=6, tiles='cartodbpositron')

cv_clipped = cv_rainfall.clip(kenya_boundary)
map_id = cv_clipped.getMapId(cv_vis)
folium.TileLayer(
    tiles=map_id['tile_fetcher'].url_format,
    attr='Rainfall Variability (CV)',
    name='Rainfall CV (%)',
    overlay=True
).add_to(Map4)

folium.GeoJson(
    kenya_geojson,
    name='Counties',
    style_function=lambda x: {'color': 'black', 'fill': False, 'weight': 1}
).add_to(Map4)

colormap_cv = LinearColormap(
    colors=['green', 'yellow', 'orange', 'red'],
    vmin=0,
    vmax=50,
    caption='Rainfall Coefficient of Variation (%)'
)
colormap_cv.add_to(Map4)

folium.LayerControl().add_to(Map4)
Map4

# %% [markdown]
# ## 🔟 Aggregate Statistics by County

# %%
def extract_county_stats(image, feature_collection, scale=5000):
    """
    Extract statistics for each county from an image.
    """
    stats = image.reduceRegions(
        collection=feature_collection,
        reducer=ee.Reducer.mean().combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        ).combine(
            reducer2=ee.Reducer.min(),
            sharedInputs=True
        ).combine(
            reducer2=ee.Reducer.max(),
            sharedInputs=True
        ),
        scale=scale
    )
    return stats

# Extract statistics for mean seasonal rainfall
print("📊 Extracting county-level statistics...")

mean_rainfall_stats = extract_county_stats(
    mean_seasonal_rainfall,
    kenya_admin1,
    scale=5000
)

# Extract CV statistics
cv_stats = extract_county_stats(
    cv_rainfall,
    kenya_admin1,
    scale=5000
)

# Extract trend statistics
trend_stats = extract_county_stats(
    rainfall_trend.select('scale'),
    kenya_admin1,
    scale=5000
)

print("✅ County statistics extracted")

# %% [markdown]
# ## 1️⃣1️⃣ Compute NDVI Statistics

# %%
# Compute mean NDVI per season
mean_ndvi = modis_ndvi.mean().multiply(0.0001).updateMask(cropland_mask)  # Scale factor
std_ndvi = modis_ndvi.reduce(ee.Reducer.stdDev()).multiply(0.0001).updateMask(cropland_mask)

# Extract NDVI statistics by county
ndvi_stats = extract_county_stats(
    mean_ndvi,
    kenya_admin1,
    scale=5000
)

print("✅ NDVI statistics computed")

# Visualize mean NDVI
ndvi_vis = {
    'min': 0,
    'max': 0.8,
    'palette': ['#d73027', '#fee08b', '#d9ef8b', '#66bd63', '#1a9850']
}

Map5 = folium.Map(location=[0.0236, 37.9062], zoom_start=6, tiles='cartodbpositron')

ndvi_clipped = mean_ndvi.clip(kenya_boundary)
map_id = ndvi_clipped.getMapId(ndvi_vis)
folium.TileLayer(
    tiles=map_id['tile_fetcher'].url_format,
    attr='Mean NDVI (Cropland)',
    name='Mean NDVI',
    overlay=True
).add_to(Map5)

folium.GeoJson(
    kenya_geojson,
    name='Counties',
    style_function=lambda x: {'color': 'black', 'fill': False, 'weight': 1}
).add_to(Map5)

colormap_ndvi = LinearColormap(
    colors=['#d73027', '#fee08b', '#d9ef8b', '#66bd63', '#1a9850'],
    vmin=0,
    vmax=0.8,
    caption='Mean NDVI'
)
colormap_ndvi.add_to(Map5)

folium.LayerControl().add_to(Map5)
Map5

# %% [markdown]
# ## 1️⃣2️⃣ Create Combined Reliability Index

# %%
def compute_reliability_index(rainfall_cv, ndvi_mean, lgp_std):
    """
    Compute a combined reliability index.
    Lower CV + Higher NDVI + Lower LGP variability = More reliable

    Index = (100 - CV) * NDVI * (100 - LGP_CV)
    Normalized to 0-100
    """
    # Invert CV (lower is better)
    cv_score = ee.Image(100).subtract(rainfall_cv).clamp(0, 100)

    # Scale NDVI to 0-100
    ndvi_score = ndvi_mean.multiply(100).clamp(0, 100)

    # Combine scores (geometric mean)
    reliability = cv_score.multiply(ndvi_score).sqrt()

    return reliability.clamp(0, 100)

reliability_index = compute_reliability_index(
    cv_rainfall,
    mean_ndvi,
    std_seasonal_rainfall
)

# Extract reliability by county
reliability_stats = extract_county_stats(
    reliability_index,
    kenya_admin1,
    scale=5000
)

print("✅ Rainfall reliability index computed")

# Visualize reliability index
reliability_vis = {
    'min': 0,
    'max': 100,
    'palette': ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']
}

Map6 = folium.Map(location=[0.0236, 37.9062], zoom_start=6, tiles='cartodbpositron')

reliability_clipped = reliability_index.clip(kenya_boundary)
map_id = reliability_clipped.getMapId(reliability_vis)
folium.TileLayer(
    tiles=map_id['tile_fetcher'].url_format,
    attr='Rainfall Reliability Index',
    name='Reliability Index',
    overlay=True
).add_to(Map6)

folium.GeoJson(
    kenya_geojson,
    name='Counties',
    style_function=lambda x: {'color': 'black', 'fill': False, 'weight': 1}
).add_to(Map6)

colormap_rel = LinearColormap(
    colors=['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850'],
    vmin=0,
    vmax=100,
    caption='Rainfall Reliability Index (0-100)'
)
colormap_rel.add_to(Map6)

folium.LayerControl().add_to(Map6)
Map6

# %% [markdown]
# ## 1️⃣3️⃣ Export Results to CSV

# %%
# Combine all statistics into one feature collection
def add_rainfall_stats(feature):
    """Add rainfall statistics to each county feature"""
    county_name = feature.get('ADM1_NAME')

    # Get stats from different collections
    rain_stats = mean_rainfall_stats.filter(ee.Filter.eq('ADM1_NAME', county_name)).first()
    cv_info = cv_stats.filter(ee.Filter.eq('ADM1_NAME', county_name)).first()
    ndvi_info = ndvi_stats.filter(ee.Filter.eq('ADM1_NAME', county_name)).first()
    rel_info = reliability_stats.filter(ee.Filter.eq('ADM1_NAME', county_name)).first()

    return feature.set({
        'mean_rainfall': rain_stats.get('mean'),
        'rainfall_cv': cv_info.get('mean'),
        'mean_ndvi': ndvi_info.get('mean'),
        'reliability_index': rel_info.get('mean')
    })

# Create final export collection
export_collection = kenya_admin1.map(add_rainfall_stats)

# Export to Google Drive
task = ee.batch.Export.table.toDrive(
    collection=export_collection,
    description='Kenya_Rainfall_Reliability_Stats',
    fileFormat='CSV',
    selectors=['ADM1_NAME', 'ADM0_NAME', 'mean_rainfall', 'rainfall_cv',
               'mean_ndvi', 'reliability_index']
)

task.start()

print("✅ Export task started!")
print("📁 Check your Google Drive for 'Kenya_Rainfall_Reliability_Stats.csv'")
print(f"Task ID: {task.id}")
print(f"Task Status: {task.status()}")

# %% [markdown]
# ## 1️⃣4️⃣ Export Time Series Data

# %%
# Export yearly rainfall time series for each county
def extract_yearly_series(year, season):
    """Extract rainfall for a specific year and season"""
    img = seasonal_rainfall_masked \
        .filter(ee.Filter.eq('year', year)) \
        .filter(ee.Filter.eq('season', season)) \
        .first()

    stats = img.reduceRegions(
        collection=kenya_admin1,
        reducer=ee.Reducer.mean(),
        scale=5000
    )

    return stats.map(lambda f: f.set('year', year, 'season', season))

# Create time series collection
time_series_list = []
for year in years:
    for season in seasons:
        ts = extract_yearly_series(year, season)
        time_series_list.append(ts)

# Flatten the list
time_series_collection = ee.FeatureCollection(time_series_list).flatten()

# Export time series
task2 = ee.batch.Export.table.toDrive(
    collection=time_series_collection,
    description='Kenya_Rainfall_TimeSeries',
    fileFormat='CSV',
    selectors=['ADM1_NAME', 'year', 'season', 'mean']
)

task2.start()

print("✅ Time series export task started!")
print("📁 Check your Google Drive for 'Kenya_Rainfall_TimeSeries.csv'")
print(f"Task ID: {task2.id}")

# %% [markdown]
# ## 1️⃣5️⃣ Summary Statistics

# %%
print("\n" + "="*60)
print("📊 RAINFALL RELIABILITY EXPLORER - SUMMARY")
print("="*60)
print(f"Analysis Period: {start_date} to {end_date}")
print(f"Number of Counties: {county_count}")
print(f"Years Analyzed: {len(years)}")
print(f"Seasons: {', '.join(seasons)}")
print("\nDatasets Used:")
print("  🌧️  CHIRPS Daily Precipitation")
print("  🌿 MODIS NDVI (MOD13A2)")
print("  🌾 GFSAD1000 Cropland Mask")
print("\nMetrics Computed:")
print("  ✓ Mean Seasonal Rainfall")
print("  ✓ Rainfall Coefficient of Variation (CV)")
print("  ✓ Rainfall Trends (Linear Fit)")
print("  ✓ Mean NDVI (Cropland Only)")
print("  ✓ Length of Growing Period Proxy")
print("  ✓ Combined Reliability Index")
print("\nExports:")
print("  📁 Kenya_Rainfall_Reliability_Stats.csv")
print("  📁 Kenya_Rainfall_TimeSeries.csv")
print("\n" + "="*60)
print("✅ WORKFLOW COMPLETE!")
print("="*60)
print("\n💡 Next Steps:")
print("1. Download CSV files from Google Drive")
print("2. Import into Observable for interactive visualization")
print("3. Create choropleth maps showing reliability patterns")
print("4. Build time series charts for trend analysis")
print("5. Add correlation analysis between rainfall and NDVI")
print("="*60)

✅ Earth Engine initialized successfully!
📍 Loaded 8 counties in Kenya
📅 Analysis period: 2010-01-01 to 2023-12-31
🌧️  CHIRPS: 5112 daily images loaded
🌿 MODIS NDVI: 322 images loaded
🌾 Cropland mask loaded
✅ Computed seasonal rainfall for 14 years × 2 seasons
📊 Onset computation function defined (simplified version)
✅ LGP proxy computed for all years and seasons
✅ Cropland mask applied to all datasets
✅ Computed rainfall variability and trends
📊 Extracting county-level statistics...
✅ County statistics extracted
✅ NDVI statistics computed
✅ Rainfall reliability index computed
✅ Export task started!
📁 Check your Google Drive for 'Kenya_Rainfall_Reliability_Stats.csv'
Task ID: DSQN62DGS5XQNVE2SWVM7HKH
Task Status: {'state': 'READY', 'description': 'Kenya_Rainfall_Reliability_Stats', 'priority': 100, 'creation_timestamp_ms': 1760125360400, 'update_timestamp_ms': 1760125360400, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'DSQN62DGS5XQNVE2SWVM7HKH', 'name': 'projects/ee-v